# Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import os
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.model_selection import HalvingGridSearchCV
import joblib


import warnings
warnings.filterwarnings('ignore')

# Loading the Dataset

In [3]:
Path_Data = '3_merged_data3.txt'

In [ ]:
df = pd.read_csv(f"{Path_Data}", sep='\t')
df.head()

# Exploratory Data Analysis (EDA)

In [5]:
df.shape

(149, 33050)

In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    print("\nColumns with missing values:")
    print(missing)
else:
    print("\nNo missing values found.")


No missing values found.


In [ ]:
# Distribution of Target Variable
plt.figure(figsize=(8, 5))
sns.histplot(df['avg7_calingiri'], bins= 30, kde = True)
plt.title('Distribution of Target Variable')
plt.xlabel('Disease Score')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Target Variable Stats
print('\nTarget Variable Stats\n')
Stats = df['avg7_calingiri'].describe()
print(Stats)

In [ ]:
# Binary feature summary (presence/absence counts)
feature_columns = df.columns[2:]  # Skip ID and target
binary_summary = df[feature_columns].agg(['sum', 'mean', 'std']).T
binary_summary.columns = ['Total_Presence', 'Presence_Rate', 'StdDev']
print("\nSample Binary Feature Summary\n")
print(binary_summary)

In [ ]:
# Select only binary feature columns (skip ID and target)
feature_columns = df.columns[2:]
binary_data = df[feature_columns]

# Calculate the mean of each column (i.e., proportion of 1s for each feature)
feature_presence_rate = binary_data.mean(axis=0)

# Average across all features
avg_1s = feature_presence_rate.mean()
avg_0s = 1 - avg_1s

# Print results
print(f"Average proportion of 1s across all binary features: {avg_1s:.4f}")
print(f"Average proportion of 0s across all binary features: {avg_0s:.4f}")


The following histogram shows how many features are rare (near 0), common (near 1), or balanced (around 0.5).
You might decide to drop features that are too rare or constant.

In [ ]:
# Histogram to Show Feature Presence
feature_columns = df.columns[2:]
binary_data = df[feature_columns]
presence_rates = binary_data.mean(axis=0)

# Plot Figure
plt.figure(figsize=(12, 6))
sns.histplot(presence_rates, bins=50, kde= True, )
plt.xlabel("Distribution of Feature Presense Rates")
plt.ylabel('No. of Features')
plt.grid()
plt.show()

 - The following HeatMap reveals redundancy or high correlation among binary features which is very useful for dimensionality reduction,
 - e.g., dropping one of two highly correlated proteins.

In [ ]:
# Find top 20 features with highest standard deviation (i.e., most varied across samples)
top_20_features = binary_data.std().sort_values(ascending=False).head(20).index
top_20_data = binary_data[top_20_features]

# Compute correlation matrix
corr_matrix = top_20_data.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title("Correlation Heatmap of Top 20 Variable Binary Features")
plt.tight_layout()
plt.show()


## Splitting Data intro Train/ Test

In [13]:
# Separate features and target
X = df.drop(columns=['ID', 'avg7_calingiri'])
y = df['avg7_calingiri']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,  random_state=42
)

print(f"X_train shape: {X_train.shape}")  # 80%
print(f"X_test shape: {X_test.shape}")
    # 20%


X_train shape: (119, 33048)
X_test shape: (30, 33048)


# Grid Search setup

In [ ]:

def get_models():
    return {
        "Linear Regression": LinearRegression(n_jobs=1),
        "Decision Tree": DecisionTreeRegressor(),
        #"Random Forest": RandomForestRegressor(n_jobs=1),
        "Gradient Boosting": GradientBoostingRegressor(),
        "Lasso Regression": Lasso(),
        #"XGBoost": XGBRegressor(use_label_encoder=False, eval_metric='rmse', n_jobs=1),
        #"LightGBM": LGBMRegressor(device='gpu', n_jobs=1),
        #"CatBoost": CatBoostRegressor(task_type="GPU", devices='0', thread_count=1, verbose=0)
        }

# Define scoring metrics
def get_scoring():
    return {
        'MSE': make_scorer(mean_squared_error, greater_is_better=False),
        'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
        'R2': make_scorer(r2_score)
    }

# Define grid search setup per model
def create_grid_searches(X, y, param_grids, k_values):
    searches = {}

    for model_name, model in get_models().items():
        for k in k_values:
            cv = KFold(n_splits=k, shuffle=True, random_state=42)
            search = HalvingGridSearchCV(
                estimator=model,
                param_grid=param_grids.get(model_name, {}),
                scoring='r2',
                cv=cv,
                n_jobs=-1,
                verbose=2,
                factor=2
            )
            key = f"{model_name} (K={k})"
            searches[key] = search
    return searches


### Define parameter Grid

In [ ]:
param_grids = {
    "Linear Regression": {},
    "Decision Tree": {
        "max_depth": [3, 5, 7, None]
    },
    # "Random Forest": {
    #     "n_estimators": [50, 100],
    #     "max_depth": [5, 10]
    # },
    "Gradient Boosting": {
        "n_estimators": [50, 100],
        "learning_rate": [0.01, 0.1]
    },
    "Lasso Regression": {
        "alpha": [0.01, 0.1, 1.0, 10.0]
    }
    # "XGBoost": {
    #     "n_estimators": [50, 100],
    #     "max_depth": [3, 5],
    #     "learning_rate": [0.01, 0.1]
    # },
    # "LightGBM": {
    #     "n_estimators": [50, 100],
    #     "learning_rate": [0.01, 0.1],
    #     "num_leaves": [31, 50]
    # },
    # "CatBoost": {
    #     "iterations": [100, 200],
    #     "learning_rate": [0.01, 0.1],
    #     "depth": [4, 6]
    # }
}

### Training Loop

In [16]:
# from tqdm import tqdm
# import pandas as pd
# import time

# # Store final results
# def train_with_progress(searches, X, y):
#     all_results = []

#     # Total tasks = total number of GridSearchCV objects
#     total_tasks = len(searches)
#     pbar = tqdm(total=total_tasks, desc="Training Models", ncols=100)

#     for name, search in searches.items():
#         print(f"\n🔍 Training: {name}")
#         start_time = time.time()

#         # Fit model
#         search.fit(X, y)

#         best_model = search.best_estimator_
#         best_params = search.best_params_
#         best_scores = search.cv_results_

#         # Predict on training set
#         y_pred = best_model.predict(X)

#         mean_mse = mean_squared_error(y, y_pred)
#         mean_mae = mean_absolute_error(y, y_pred)
#         mean_r2  = r2_score(y, y_pred)


#         all_results.append({
#             "Model": name,
#             "Best Params": best_params,
#             "MSE": mean_mse,
#             "MAE": mean_mae,
#             "R2": mean_r2,
#             "Training Time (s)": round(time.time() - start_time, 2)
#         })

#         pbar.update(1)

#     pbar.close()

#     # Save results as DataFrame
#     results_df = pd.DataFrame(all_results)
#     results_df = results_df.sort_values(by="R2", ascending=False)
#     return results_df


In [ ]:


# Store final results and save trained models
def train_with_progress(searches, X, y, save_dir="saved_models"):
    all_results = []

    # Create directory to save models
    os.makedirs(save_dir, exist_ok=True)

    total_tasks = len(searches)
    pbar = tqdm(total=total_tasks, desc="Training Models", ncols=100)

    for name, search in searches.items():
        print(f"\n🔍 Training: {name}")
        start_time = time.time()

        # Fit model
        search.fit(X, y)

        best_model = search.best_estimator_
        best_params = search.best_params_

        # Predict on training set
        y_pred = best_model.predict(X)

        mean_mse = mean_squared_error(y, y_pred)
        mean_mae = mean_absolute_error(y, y_pred)
        mean_r2  = r2_score(y, y_pred)

        # Save the model
        safe_model_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
        model_path = os.path.join(save_dir, f"{safe_model_name}.pkl")
        joblib.dump(best_model, model_path)

        # Record results
        all_results.append({
            "Model": name,
            "Best Params": best_params,
            "MSE": mean_mse,
            "MAE": mean_mae,
            "R2": mean_r2,
            "Training Time (s)": round(time.time() - start_time, 2)
        })

        pbar.update(1)

    pbar.close()

    # Save results as DataFrame
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(by="R2", ascending=False)
    return results_df


In [ ]:
k_values = [3,5,7]
# Create grid searches
searches = create_grid_searches(X_train, y_train, param_grids, k_values)

# Run training
results_df = train_with_progress(searches, X_train, y_train)


In [ ]:
# Show top results
print("\n📊 Final Leaderboard:")
print(results_df.to_string(index=False))

### Save trainings as CSV

In [24]:
# Optionally save to CSV
results_df.to_csv("training_results_halvinggrid.csv", index=False)


### Get best overall Model based on MSE, MAE and R2 on training data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def get_best_overall_model(results_df):
    df = results_df.copy()

    # Invert MSE and MAE (since lower is better, we flip them for scoring)
    df["Inv_MSE"] = -df["MSE"]
    df["Inv_MAE"] = -df["MAE"]

    # Normalize all metrics to [0, 1] range
    scaler = MinMaxScaler()
    df[["Norm_R2", "Norm_MSE", "Norm_MAE"]] = scaler.fit_transform(
        df[["R2", "Inv_MSE", "Inv_MAE"]]
    )

    # Combine all three normalized scores
    df["Combined_Score"] = df["Norm_R2"] + df["Norm_MSE"] + df["Norm_MAE"]

    # Sort by combined score
    df = df.sort_values(by="Combined_Score", ascending=False).reset_index(drop=True)

    # Print the best model's stats
    print("🏆 Best Model Based on Combined R², MSE, and MAE:\n")
    print(df.loc[0, ["Model", "R2", "MSE", "MAE", "Combined_Score"]])

    return df.loc[0]

best_model = get_best_overall_model(results_df)


### Visualization from the training dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load both CSVs
df_grid = pd.read_csv("training_results.csv")
df_halving = pd.read_csv("training_results_halvinggrid.csv")

# 2. Add method column to identify source
df_grid['Method'] = 'GridSearchCV'
df_halving['Method'] = 'HalvingGridSearchCV'

# 3. Combine into one DataFrame
df_combined = pd.concat([df_grid, df_halving], ignore_index=True)

# Optional: Strip parameter details for clearer plots
df_combined['Model Name'] = df_combined['Model'].str.extract(r'^([^\(]+)')

# 4. Set plot style
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# 5. Plot MSE Comparison
plt.figure(figsize=(12, 6))
sns.barplot(data=df_combined, x='Model Name', y='MSE', hue='Method')
plt.title("📉 MSE Comparison: GridSearchCV vs HalvingGridSearchCV")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 6. Plot MAE Comparison
plt.figure(figsize=(12, 6))
sns.barplot(data=df_combined, x='Model Name', y='MAE', hue='Method')
plt.title("📉 MAE Comparison: GridSearchCV vs HalvingGridSearchCV")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 7. Plot R² Comparison
plt.figure(figsize=(12, 6))
sns.barplot(data=df_combined, x='Model Name', y='R2', hue='Method')
plt.title("📈 R² Comparison: GridSearchCV vs HalvingGridSearchCV")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Evaluate models on testing dataset

In [27]:

def evaluate_saved_models(X_test, y_test, model_dir="saved_models", output_csv="test_results.csv"):
    results = []

    for filename in os.listdir(model_dir):
        if filename.endswith(".pkl"):
            model_path = os.path.join(model_dir, filename)
            model_name = filename.replace(".pkl", "").replace("_", " ")

            # Load model
            model = joblib.load(model_path)

            # Predict
            y_pred = model.predict(X_test)

            # Evaluate
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            results.append({
                "Model": model_name,
                "MSE": mse,
                "MAE": mae,
                "R2": r2
            })

    # Save to CSV
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="R2", ascending=False)
    results_df.to_csv(output_csv, index=False)

    print("\n✅ Test evaluation completed and saved to", output_csv)
    print(results_df)

    return results_df


In [ ]:
test_results_df = evaluate_saved_models(X_test, y_test)

### Get best overall Model based on MSE, MAE and R2 on Testing data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def get_best_overall_model(results_df):
    df = results_df.copy()

    # Invert MSE and MAE (since lower is better, we flip them for scoring)
    df["Inv_MSE"] = -df["MSE"]
    df["Inv_MAE"] = -df["MAE"]

    # Normalize all metrics to [0, 1] range
    scaler = MinMaxScaler()
    df[["Norm_R2", "Norm_MSE", "Norm_MAE"]] = scaler.fit_transform(
        df[["R2", "Inv_MSE", "Inv_MAE"]]
    )

    # Combine all three normalized scores
    df["Combined_Score"] = df["Norm_R2"] + df["Norm_MSE"] + df["Norm_MAE"]

    # Sort by combined score
    df = df.sort_values(by="Combined_Score", ascending=False).reset_index(drop=True)

    # Print the best model's stats
    print("🏆 Best Model Based on Combined R², MSE, and MAE:\n")
    print(df.loc[0, ["Model", "R2", "MSE", "MAE", "Combined_Score"]])

    return df.loc[0]

best_model = get_best_overall_model(test_results_df)


# Visualize the results on testing dataset

In [ ]:
# Set up the visual theme
plt.figure(figsize=(12, 6))
sns.set(style="whitegrid")

# Plot R²
sns.barplot(x="R2", y="Model", data=test_results_df, palette="viridis")
plt.title("📈 Test R² Scores by Model")
plt.xlabel("R² Score")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

# Plot MAE and MSE (side-by-side)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(x="MAE", y="Model", data=test_results_df, ax=axes[0], palette="mako")
axes[0].set_title("📉 MAE on Test Set")

sns.barplot(x="MSE", y="Model", data=test_results_df, ax=axes[1], palette="rocket")
axes[1].set_title("📉 MSE on Test Set")

plt.tight_layout()
plt.show()
